In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002A4B3F03620>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002A4B4F42E40>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [7]:
from pydantic import BaseModel,fields

from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")
    genre:str=Field(description="The genre of the movie")
    
    
model_with_structure=model.with_structured_output(Movie,include_raw=True)
model_with_structure
    

{
  raw: RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002A4B3F03620>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002A4B4F42E40>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies rating out of 10', 'type': 'number'}, 'genre': {'description'

In [3]:
model.invoke("Provide details about the moview Inception")

AIMessage(content='<think>\nOkay, so I need to provide details about the movie Inception. Let me start by recalling what I know about the movie. I remember it\'s directed by Christopher Nolan, and he\'s known for other films like The Dark Knight and Interstellar. The main character is Dom Cobb, played by Leonardo DiCaprio. The movie involves some kind of dream manipulation, right? Like entering people\'s dreams to plant ideas. \n\nThe title "Inception" probably refers to the act of planting an idea in someone\'s subconscious. I think there\'s a concept called "dream sharing" or "shared dreaming" where people can enter each other\'s dreams. The plot might involve a team of individuals with different roles, like a thief who steals secrets and a dream architect who designs the dream environments. There\'s also a character who can erase people\'s memories, which is part of the process to ensure the target doesn\'t remember the dream. \n\nI recall that the movie has a complex narrative stru

In [8]:
response=model_with_structure.invoke("Provide details about the moview Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me check if I have the right tool for that. The available tool is the Movie function, which requires title, year, director, rating, and genre. I need to make sure I have all that information for Inception. Let me recall: Inception was directed by Christopher Nolan, released in 2010. The genre is science fiction or action. The rating is probably around 8.8 on IMDb. I need to structure the function call with these details. Wait, the year is 2010, correct? Yes. The title is "Inception". Director is definitely Christopher Nolan. Genre could be listed as Science Fiction or Action. Rating is 8.8. Let me put that all together in the JSON object as specified. Make sure all required fields are included. Yep, that\'s all there. No need for extra fields. Alright, that should do it.\n', 'tool_calls': [{'id': 'nnhvt54j4', 'function': {'arguments': '{"director":"

In [13]:
from pydantic import BaseModel,Field


class Actor(BaseModel):
    name:str
    role:str
    
class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genre:list[str]
    budget:float|None=Field(default=None,description="The budget of the movie in millions USD")

In [16]:
model_with_structure=model.with_structured_output(MovieDetails)
response=model_with_structure.invoke("Provide details about the moview Inception")
response.dict()

C:\Users\kakab\AppData\Local\Temp\ipykernel_31108\1077484331.py:3: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  response.dict()


{'title': 'Inception',
 'year': 2010,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'},
  {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'},
  {'name': 'Ellen Page', 'role': 'Ariadne'},
  {'name': 'Tom Hardy', 'role': 'Mr. Freeze'}],
 'genre': ['Science Fiction', 'Action'],
 'budget': 160.0}

In [15]:
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie "Inception". Let me see what functions I have available. There\'s a MovieDetails function that requires title, year, cast, and genre. The parameters also include budget, which is optional.\n\nFirst, I need to recall the information about Inception. The title is "Inception", released in 2010. The director is Christopher Nolan. The main cast includes Leonardo DiCaprio as Dom Cobb, Joseph Gordon-Levitt as Arthur, Ellen Page as Ariadne, and Tom Hardy as Johan. The genres are Science Fiction and Action. The budget was around $160 million. \n\nWait, the function requires the cast as an array of objects with name and role. So I need to structure each cast member with their respective roles. Also, genres should be an array of strings. Let me make sure all required fields are included: title, year, cast, genre. Budget is optional, but I can include it since I know the figu

## TypeDict

In [21]:
from typing_extensions import TypedDict,Annotated


class Movie(TypedDict):
    title:Annotated[str,...,"this is the title of the movie"]
    year:Annotated[int,...,"the year the movie was released"]
    director:Annotated[str,...,"the director of the movie"]
    rating:Annotated[float,...,"the movies rating out of 10"]
    genre:Annotated[str,...,"the genre of the movie"]
    actor:Annotated[list[str],...,"the main actors in the movie"]
    role:Annotated[list[str],...,"the role of the main actors in the movie"]
    
    
model_with_structure=model.with_structured_output(Movie)
response=model_with_structure.invoke("Provide details about the moview avenger end game")
response
    

{'actor': ['Robert Downey Jr.',
  'Chris Evans',
  'Mark Ruffalo',
  'Chris Hemsworth',
  'Scarlett Johansson',
  'Jeremy Renner'],
 'director': 'Anthony Russo, Joe Russo',
 'genre': 'Superhero, Action',
 'rating': 8.4,
 'role': ['Tony Stark/Iron Man',
  'Steve Rogers/Captain America',
  'Bruce Banner/Hulk',
  'Thor',
  'Natasha Romanoff/Black Widow',
  'Clint Barton/Hawkeye'],
 'title': 'Avengers: Endgame',
 'year': 2019}

In [22]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

## DataClass

In [27]:
from pydantic import BaseModel,Field
from langchain.agents import create_agent

class ConstactInfo(BaseModel):
    name:str=Field(description="The name of the contact")
    email:str=Field(description="The email address of the contact")
    phone:str|None=Field(default=None,description="The phone number of the contact")
    
    

agents=create_agent(model="groq:qwen/qwen3-32b",response_format=ConstactInfo)

responseagents=agents.invoke(
    {"messages":
    
    [{"role":"user","content":"Provide contact information from:John Doe ,abc@xyz.com, 123-456-7890"}]
    }
     )




In [32]:
responseagents["structured_response"].dict()

C:\Users\kakab\AppData\Local\Temp\ipykernel_31108\126005708.py:1: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  responseagents["structured_response"].dict()


{'name': 'John Doe', 'email': 'abc@xyz.com', 'phone': '123-456-7890'}

In [33]:
responseagents

{'messages': [HumanMessage(content='Provide contact information from:John Doe ,abc@xyz.com, 123-456-7890', additional_kwargs={}, response_metadata={}, id='0e556ccc-f8df-46c9-9ca0-d1f09aed76ef'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user wants me to provide contact information based on the input they gave: "John Doe ,abc@xyz.com, 123-456-7890". First, I need to parse this information correctly. The input has three parts separated by commas. The first part is the name, which is "John Doe". The second part is the email address, "abc@xyz.com". The third part is the phone number, "123-456-7890".\n\nNow, looking at the function provided, it\'s called "ConstactInfo" (I notice there might be a typo here, maybe it\'s supposed to be "ContactInfo"? But I should follow the given name). The function requires the parameters name, email, and phone. The required fields are name and email, and phone is optional, which can be a string or null. The user p

In [34]:
from pydantic import BaseModel,Field
from langchain.agents import create_agent

class ConstactInfo(TypedDict):
    name:str
    email:str
    phone:str
    
    

agents=create_agent(model="groq:qwen/qwen3-32b",response_format=ConstactInfo)

responseagents=agents.invoke(
    {"messages":
    
    [{"role":"user","content":"Provide contact information from:John Doe ,abc@xyz.com, 123-456-7890"}]
    }
     )

In [35]:
responseagents

{'messages': [HumanMessage(content='Provide contact information from:John Doe ,abc@xyz.com, 123-456-7890', additional_kwargs={}, response_metadata={}, id='2a5c26a3-c30d-40a2-8981-56aca53a91f9'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user wants me to provide contact information from the given details: John Doe, abc@xyz.com, 123-456-7890. I need to use the ConstactInfo function they mentioned. Wait, the function name might be a typo. Let me check the tools provided again.\n\nLooking at the tools, the function is called "ConstactInfo". Hmm, that\'s probably a typo for "ContactInfo". But I should follow the given name exactly. The parameters required are name, email, and phone. The user provided all three pieces of information. So I need to map each part to the correct parameter. Name is John Doe, email is abc@xyz.com, and phone is 123-456-7890. I\'ll structure the JSON with these values. Make sure the keys are correct: "name", "email", "pho

In [38]:
responseagents["structured_response"]

{'name': 'John Doe', 'email': 'abc@xyz.com', 'phone': '123-456-7890'}